In [2]:
import datasets

In [3]:
from pathlib import Path

In [4]:
! ls -l multitask/huggingface_format/

total 112
drwxr-xr-x@ 6 L069561  staff    192 Sep 23 07:21 2010_i2b2_va
drwxr-xr-x@ 4 L069561  staff    128 Sep 23 09:48 2010_i2b2_va_ner_only
drwxr-xr-x@ 4 L069561  staff    128 Sep 23 09:48 2010_i2b2_va_re_only
drwxr-xr-x@ 6 L069561  staff    192 Sep 23 09:48 2010_i2b2_va_section
drwxr-xr-x@ 6 L069561  staff    192 Sep 21 19:51 2018_n2c2_track1
drwxr-xr-x@ 6 L069561  staff    192 Sep 21 19:51 2018_n2c2_track2
drwxr-xr-x@ 4 L069561  staff    128 Sep 22 16:48 2018_n2c2_track2_ner_only
drwxr-xr-x@ 6 L069561  staff    192 Sep 22 22:18 2018_n2c2_track2_paragraph
drwxr-xr-x@ 4 L069561  staff    128 Sep 22 16:48 2018_n2c2_track2_re_only
drwxr-xr-x@ 6 L069561  staff    192 Sep 21 19:51 2019_n2c2_track1
drwxr-xr-x@ 6 L069561  staff    192 Sep 21 21:23 2019_n2c2_track1_cls
drwxr-xr-x@ 6 L069561  staff    192 Sep 21 19:51 2019_n2c2_track2
drwxr-xr-x@ 6 L069561  staff    192 Sep 21 16:43 2022n2c2_track1
drwxr-xr-x@ 4 L069561  staff    128 Sep 22 16:48 2022n2c2_track1_attr_only
drwxr-xr-x@ 4 L069

In [5]:
root = Path("multitask/huggingface_format/")

## merge data

In [50]:
from collections import defaultdict
from datasets import load_from_disk, concatenate_datasets, DatasetDict

target_datasets = [
    (root / "2018_n2c2_track1").as_posix(),
    (root / "2018_n2c2_track2").as_posix(),
    (root / "2018_n2c2_track2_paragraph").as_posix(),
    (root / "2019_n2c2_track1_cls").as_posix(),
    (root / "2019_n2c2_track2").as_posix(),
    (root / "2022n2c2_track1").as_posix(),
    (root / "2022n2c2_track2_mimic").as_posix(),
    (root / "2022n2c2_track2_uw").as_posix(),
    (root / "MedNLI").as_posix(),
    (root / "MedQA-USMLE-US").as_posix(),
    (root / "umn_abbreviation").as_posix(),
]


SPLIT_MAP = {"train": "train", "dev": "dev", "validation": "dev"}

buckets = defaultdict(list)
for path in target_datasets:
    for split_name, split in load_from_disk(path).items():
        target = SPLIT_MAP.get(split_name.lower())
        if target:
            buckets[target].append(split)

target_dataset_for_training = DatasetDict({
    name: concatenate_datasets(parts).shuffle(seed=42)
    for name, parts in buckets.items()
})

target_dataset_for_training

DatasetDict({
    train: Dataset({
        features: ['messages', 'task', 'category', 'dataset', 'id'],
        num_rows: 51720
    })
    dev: Dataset({
        features: ['messages', 'task', 'category', 'dataset', 'id'],
        num_rows: 8425
    })
})

In [54]:
output_path = (root / "med_flan").as_posix()
target_dataset_for_training.save_to_disk(output_path)

Saving the dataset (0/1 shards):   0%|          | 0/51720 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/8425 [00:00<?, ? examples/s]

## 2018 n2c2n track 1

In [7]:
ds1 = datasets.load_from_disk((root / "2018_n2c2_track1").as_posix())

In [8]:
ds1

DatasetDict({
    train: Dataset({
        features: ['messages', 'task', 'category', 'dataset', 'id'],
        num_rows: 2626
    })
    test: Dataset({
        features: ['messages', 'task', 'category', 'dataset', 'id'],
        num_rows: 1118
    })
    dev: Dataset({
        features: ['messages', 'task', 'category', 'dataset', 'id'],
        num_rows: 335
    })
})

In [9]:
ds1["train"][1]

{'messages': [{'content': '<<Classification: Cohort_Eligibility>>\nYou are performing text classification: read the input and assign it the label(s) described in the task-specific instructions that follow, and return them as the JSON object those instructions specify.',
   'role': 'system'},
  {'content': 'Given the patient\'s longitudinal clinical notes and ONE named clinical trial eligibility criterion (with its definition), decide whether the patient meets that single criterion. Return JSON {"label": "met"} or {"label": "not met"}.\nINPUT: Criterion: ADVANCED-CAD\nDefinition: Advanced cardiovascular disease (CAD), defined as at least 2 of: taking 2+ CAD medications, history of myocardial infarction, currently experiencing angina, or ischemia past/present.\n\nPatient notes:\n\n\nRecord date: 2106-02-12\n\nCampbell Orthopedic Associates\n4 Madera Circle\nOmak, GA 28172\n \nHabib Valenzuela, M.D.\n \n \n                                             Valdez, Harlan Jr.  \n                

In [10]:
ds1["test"][0]

{'messages': [{'content': '<<Classification: Cohort_Eligibility>>\nYou are performing text classification: read the input and assign it the label(s) described in the task-specific instructions that follow, and return them as the JSON object those instructions specify.',
   'role': 'system'},
  {'content': 'Given the patient\'s longitudinal clinical notes and ONE named clinical trial eligibility criterion (with its definition), decide whether the patient meets that single criterion. Return JSON {"label": "met"} or {"label": "not met"}.\nINPUT: Criterion: ABDOMINAL\nDefinition: History of intra-abdominal surgery, small/large intestine resection, or small bowel obstruction.\n\nPatient notes:\n\n\nRecord date: 2135-07-10\n\n\n\n07/10/2135\n\n\n\n\nQuincy Rucker, M.D. \n03 Catalina Lane\nAbilene, IN 97765\n\n\n\n\n\n\nRE: EGAN, VAL\nMRN: 8268084\nDOB: 07/30/2063\n\n\nDear Dr. Rucker, \n\n\nI had the pleasure of seeing Mr. Val in the office today. As you recall, he is a very pleasant 71-year

In [11]:
ds1["test"][1]

{'messages': [{'content': '<<Classification: Cohort_Eligibility>>\nYou are performing text classification: read the input and assign it the label(s) described in the task-specific instructions that follow, and return them as the JSON object those instructions specify.',
   'role': 'system'},
  {'content': 'Given the patient\'s longitudinal clinical notes and ONE named clinical trial eligibility criterion (with its definition), decide whether the patient meets that single criterion. Return JSON {"label": "met"} or {"label": "not met"}.\nINPUT: Criterion: ADVANCED-CAD\nDefinition: Advanced cardiovascular disease (CAD), defined as at least 2 of: taking 2+ CAD medications, history of myocardial infarction, currently experiencing angina, or ischemia past/present.\n\nPatient notes:\n\n\nRecord date: 2135-07-10\n\n\n\n07/10/2135\n\n\n\n\nQuincy Rucker, M.D. \n03 Catalina Lane\nAbilene, IN 97765\n\n\n\n\n\n\nRE: EGAN, VAL\nMRN: 8268084\nDOB: 07/30/2063\n\n\nDear Dr. Rucker, \n\n\nI had the ple

## 2018 n2c2 track2

In [13]:
ds2 = datasets.load_from_disk((root / "2018_n2c2_track2").as_posix())
ds2

DatasetDict({
    train: Dataset({
        features: ['messages', 'task', 'category', 'dataset', 'id'],
        num_rows: 303
    })
    test: Dataset({
        features: ['messages', 'task', 'category', 'dataset', 'id'],
        num_rows: 202
    })
    dev: Dataset({
        features: ['messages', 'task', 'category', 'dataset', 'id'],
        num_rows: 61
    })
})

In [14]:
ds2["train"][0]

{'messages': [{'content': '<<Information_Extraction: NER_RE>>\nYou are performing information extraction from clinical or biomedical text: identify the entities, events, relations, or attributes described in the task-specific instructions that follow, and return them as the JSON object those instructions specify.',
   'role': 'system'},
  {'content': 'Task: Extract every Drug mention in this note and its linked clinical attributes, per the 2018 n2c2 Track 2 ADE/medication-extraction schema (Henry et al. 2020, JAMIA).\nDefinitions: Drug = a medication name (generic, brand, or drug class). Strength = amount of active ingredient per dose unit (e.g. "40 mg"). Form = the drug\'s physical form (e.g. tablet, patch, cream). Dosage = amount given per administration (e.g. "2 tablets"). Frequency = how often it\'s taken (e.g. "twice daily"). Route = how it enters the body (e.g. "PO", "IV", "topically"). Duration = how long it is/was taken. Reason = the medical indication for the drug. ADE = an ad

In [15]:
ds2["test"][0]

{'messages': [{'content': '<<Information_Extraction: NER_RE>>\nYou are performing information extraction from clinical or biomedical text: identify the entities, events, relations, or attributes described in the task-specific instructions that follow, and return them as the JSON object those instructions specify.',
   'role': 'system'},
  {'content': 'Task: Extract every Drug mention in this note and its linked clinical attributes, per the 2018 n2c2 Track 2 ADE/medication-extraction schema (Henry et al. 2020, JAMIA).\nDefinitions: Drug = a medication name (generic, brand, or drug class). Strength = amount of active ingredient per dose unit (e.g. "40 mg"). Form = the drug\'s physical form (e.g. tablet, patch, cream). Dosage = amount given per administration (e.g. "2 tablets"). Frequency = how often it\'s taken (e.g. "twice daily"). Route = how it enters the body (e.g. "PO", "IV", "topically"). Duration = how long it is/was taken. Reason = the medical indication for the drug. ADE = an ad

## 2022 n2c2 track1

In [17]:
ds3 = datasets.load_from_disk((root / "2022n2c2_track1/").as_posix())
ds3

DatasetDict({
    train: Dataset({
        features: ['messages', 'task', 'category', 'dataset', 'id'],
        num_rows: 350
    })
    dev: Dataset({
        features: ['messages', 'task', 'category', 'dataset', 'id'],
        num_rows: 50
    })
    test: Dataset({
        features: ['messages', 'task', 'category', 'dataset', 'id'],
        num_rows: 100
    })
})

In [18]:
ds3["train"][0]

{'messages': [{'content': '<<Information_Extraction: Medication_Event>>\nYou are performing information extraction from clinical or biomedical text: identify the entities, events, relations, or attributes described in the task-specific instructions that follow, and return them as the JSON object those instructions specify.',
   'role': 'system'},
  {'content': 'Task: For each medication mention, classify whether a medication change is discussed, per the CMED schema (Mahajan et al. 2021).\nDefinitions: Disposition = a change is discussed (e.g. "start Plavix"). NoDisposition = no change is discussed (e.g. "continue lisinopril"). Undetermined = unclear from local context (e.g. "Plan: Lasix"). For each Disposition mention, also classify: Action -- Start/Stop/Increase/Decrease/UniqueDose/OtherChange/Unknown; Certainty -- Certain/Hypothetical/Conditional/Unknown; Temporality -- Past/Present/Future/Unknown; Actor -- Physician/Patient/Unknown; Negation -- Negated/NotNegated. A single mention m

In [19]:
ds3["test"][0]

{'messages': [{'content': '<<Information_Extraction: Medication_Event>>\nYou are performing information extraction from clinical or biomedical text: identify the entities, events, relations, or attributes described in the task-specific instructions that follow, and return them as the JSON object those instructions specify.',
   'role': 'system'},
  {'content': 'Task: For each medication mention, classify whether a medication change is discussed, per the CMED schema (Mahajan et al. 2021).\nDefinitions: Disposition = a change is discussed (e.g. "start Plavix"). NoDisposition = no change is discussed (e.g. "continue lisinopril"). Undetermined = unclear from local context (e.g. "Plan: Lasix"). For each Disposition mention, also classify: Action -- Start/Stop/Increase/Decrease/UniqueDose/OtherChange/Unknown; Certainty -- Certain/Hypothetical/Conditional/Unknown; Temporality -- Past/Present/Future/Unknown; Actor -- Physician/Patient/Unknown; Negation -- Negated/NotNegated. A single mention m

## 2022 n2c2 track 2 - mimic

In [20]:
ds4 = datasets.load_from_disk((root / "2022n2c2_track2_mimic").as_posix())
ds4

DatasetDict({
    train: Dataset({
        features: ['messages', 'task', 'category', 'dataset', 'id'],
        num_rows: 1316
    })
    dev: Dataset({
        features: ['messages', 'task', 'category', 'dataset', 'id'],
        num_rows: 188
    })
    test: Dataset({
        features: ['messages', 'task', 'category', 'dataset', 'id'],
        num_rows: 373
    })
})

In [21]:
ds4["train"][0]

{'messages': [{'content': '<<Information_Extraction: SDoH>>\nYou are performing information extraction from clinical or biomedical text: identify the entities, events, relations, or attributes described in the task-specific instructions that follow, and return them as the JSON object those instructions specify.',
   'role': 'system'},
  {'content': 'Task: Extract social-determinants-of-health (SDOH) events from this note, per the SHAC annotation schema (Lybarger et al., 2022 n2c2 Track 2).\nDefinitions: an SDOH event is a trigger span (Employment, LivingStatus, Tobacco, Alcohol, or Drug) plus role-labeled argument spans. Employment roles: Status (employed/unemployed/retired/on_disability/student/homemaker), Type (occupation), Duration, History. LivingStatus roles: Status (current/past/future), Type (alone/with_family/with_others/homeless), Duration, History. Tobacco/Alcohol/Drug roles: Status (none/current/past), Type, Amount, Frequency, Method, Duration, History -- Drug covers illicit

In [22]:
ds4["test"][0]

{'messages': [{'content': '<<Information_Extraction: SDoH>>\nYou are performing information extraction from clinical or biomedical text: identify the entities, events, relations, or attributes described in the task-specific instructions that follow, and return them as the JSON object those instructions specify.',
   'role': 'system'},
  {'content': 'Task: Extract social-determinants-of-health (SDOH) events from this note, per the SHAC annotation schema (Lybarger et al., 2022 n2c2 Track 2).\nDefinitions: an SDOH event is a trigger span (Employment, LivingStatus, Tobacco, Alcohol, or Drug) plus role-labeled argument spans. Employment roles: Status (employed/unemployed/retired/on_disability/student/homemaker), Type (occupation), Duration, History. LivingStatus roles: Status (current/past/future), Type (alone/with_family/with_others/homeless), Duration, History. Tobacco/Alcohol/Drug roles: Status (none/current/past), Type, Amount, Frequency, Method, Duration, History -- Drug covers illicit

## 2019 n2c2 track 1

In [23]:
ds5 = datasets.load_from_disk((root / "2019_n2c2_track1_cls").as_posix())
ds5

DatasetDict({
    train: Dataset({
        features: ['messages', 'task', 'category', 'dataset', 'id'],
        num_rows: 1654
    })
    test: Dataset({
        features: ['messages', 'task', 'category', 'dataset', 'id'],
        num_rows: 412
    })
    dev: Dataset({
        features: ['messages', 'task', 'category', 'dataset', 'id'],
        num_rows: 124
    })
})

In [24]:
ds5["train"][4]

{'messages': [{'content': '<<Classification: STS_Classification>>\nYou are performing text classification: read the input and assign it the label(s) described in the task-specific instructions that follow, and return them as the JSON object those instructions specify.',
   'role': 'system'},
  {'content': 'Classify the semantic similarity of the two clinical sentences into one of 5 categories: "dissimilar" (unrelated, or share no clinically meaningful information), "weakly_related" (same general topic but very different clinical content), "partially_equivalent" (share important clinical information but differ in other important details), "mostly_equivalent" (same essential clinical meaning, only minor/unimportant details differ), or "highly_equivalent" (semantically equivalent or nearly so). Return JSON {"category": <one of the 5 values above, as a string>}.\nINPUT: Sentence 1: The diagnosis and treatment plan were explained to the family/caregiver who expressed understanding of the in

## 2019 n2c2 track 2

In [25]:
ds6 = datasets.load_from_disk((root / "2019_n2c2_track2/").as_posix())
ds6

DatasetDict({
    train: Dataset({
        features: ['messages', 'task', 'category', 'dataset', 'id'],
        num_rows: 99
    })
    test: Dataset({
        features: ['messages', 'task', 'category', 'dataset', 'id'],
        num_rows: 115
    })
    dev: Dataset({
        features: ['messages', 'task', 'category', 'dataset', 'id'],
        num_rows: 34
    })
})

In [26]:
ds6["train"][0]

{'messages': [{'content': '<<Information_Extraction: Family_History>>\nYou are performing information extraction from clinical or biomedical text: identify the entities, events, relations, or attributes described in the task-specific instructions that follow, and return them as the JSON object those instructions specify.',
   'role': 'system'},
  {'content': 'Task: Extract family history from the note, per the 2019 n2c2 Track 2 schema (Liu et al. 2018, BioCreative/OHNLP).\nDefinitions: FamilyMember is one of Father/Mother/Parent/Sister/Brother/Daughter/Son/Child/Grandmother/Grandfather/Grandparent/Cousin/Sibling/Aunt/Uncle, each with a side of family (Paternal/Maternal/NA -- first-degree relatives are always NA). Observation is a disease/condition mentioned for a family member. LivingStatus score = alive_score * healthy_score, each in {Yes=2, NA=1, No=0} (so 4 = alive and healthy, 0 = deceased).\nOutputFormat: Return JSON with 4 keys: "family_members" (list of {"member", "side"}), "obs

## mednli

In [28]:
ds7 = datasets.load_from_disk((root / "MedNLI").as_posix())
ds7

DatasetDict({
    train: Dataset({
        features: ['messages', 'task', 'category', 'dataset', 'id'],
        num_rows: 11232
    })
    dev: Dataset({
        features: ['messages', 'task', 'category', 'dataset', 'id'],
        num_rows: 1395
    })
    test: Dataset({
        features: ['messages', 'task', 'category', 'dataset', 'id'],
        num_rows: 1422
    })
})

In [29]:
ds7["train"][0]

{'messages': [{'content': '<<Classification: NLI>>\nYou are performing text classification: read the input and assign it the label(s) described in the task-specific instructions that follow, and return them as the JSON object those instructions specify.',
   'role': 'system'},
  {'content': 'Determine the relationship between the premise and hypothesis: entailment, contradiction, or neutral. Return JSON {"label": ...}.\nINPUT: Premise: Labs were notable for Cr 1.7 (baseline 0.5 per old records) and lactate 2.4.\nHypothesis:  Patient has elevated Cr',
   'role': 'user'},
  {'content': '{"label": "entailment"}', 'role': 'assistant'}],
 'task': 'nli',
 'category': 'classification',
 'dataset': 'MedNLI',
 'id': 'mli_train_v1.jsonl#0'}

## semeval 2015

In [30]:
ds8 = datasets.load_from_disk((root / "semeval_2015/").as_posix())
ds8

DatasetDict({
    train: Dataset({
        features: ['messages', 'task', 'category', 'dataset', 'id'],
        num_rows: 298
    })
    test: Dataset({
        features: ['messages', 'task', 'category', 'dataset', 'id'],
        num_rows: 133
    })
    dev: Dataset({
        features: ['messages', 'task', 'category', 'dataset', 'id'],
        num_rows: 40
    })
})

In [31]:
ds8["train"][0]

{'messages': [{'content': '<<Information_Extraction: Disorder_Normalization>>\nYou are performing information extraction from clinical or biomedical text: identify the entities, events, relations, or attributes described in the task-specific instructions that follow, and return them as the JSON object those instructions specify.',
   'role': 'system'},
  {'content': 'Identify all disorder mentions in the clinical text and normalize each to a UMLS/SNOMED-CT CUI, or "CUI-less" if no matching concept exists. Return JSON {"entities": [{"text": ..., "cui": ...}, ...]}.\nINPUT: 16139\t||||\t98\t||||\t15836\t||||\tDISCHARGE_SUMMARY\t||||\t2015-03-24 00:00:00.0\t||||\t\t||||\t\t||||\t\t||||\t\nAdmission Date:  [**2015-03-17**]       Discharge Date:  [**2015-03-24**]\n\nDate of Birth:   [**1974-10-03**]       Sex:  F\n\nService: Neurosurgery\n\nHISTORY OF PRESENT ILLNESS:  The patient is a 40-year-old female with complaints of headache and dizziness.  In [**2015-01-14**], the patient had headac

## umn

In [32]:
ds9 = datasets.load_from_disk((root / "umn_abbreviation/").as_posix())
ds9

DatasetDict({
    train: Dataset({
        features: ['messages', 'task', 'category', 'dataset', 'id'],
        num_rows: 19257
    })
    dev: Dataset({
        features: ['messages', 'task', 'category', 'dataset', 'id'],
        num_rows: 4127
    })
    test: Dataset({
        features: ['messages', 'task', 'category', 'dataset', 'id'],
        num_rows: 4127
    })
})

In [33]:
ds9["train"][0]

{'messages': [{'content': '<<Classification: Abbreviation_Disambiguation>>\nYou are performing text classification: read the input and assign it the label(s) described in the task-specific instructions that follow, and return them as the JSON object those instructions specify.',
   'role': 'system'},
  {'content': 'Given a clinical abbreviation and its surrounding context, determine what the abbreviation stands for in this specific context. Return JSON {"expansion": ...}.\nINPUT: Abbreviation: SBP\nSection: DISCHARGE MEDICATIONS\nContext: 9. Fluphenazine 50 mg p.o. b.i.d. 10. Lamictal 25 mg p.o. each day at bedtime. 11. Synthroid 75 mcg p.o. daily. 12. Lisinopril 5 mg p.o. daily, hold if SBP less than 90. 13. Mirtazapine dissolve tabs 45 mg p.o. each day at bedtime. 14. Zyprexa 20 mg p.o. daily. 15. Protonix 40 mg p.o. daily.',
   'role': 'user'},
  {'content': '{"expansion": "systolic blood pressure"}',
   'role': 'assistant'}],
 'task': 'abbreviation_disambiguation',
 'category': 'cl

## medqa

In [34]:
ds10 = datasets.load_from_disk((root / "MedQA-USMLE-US").as_posix())
ds10

DatasetDict({
    train: Dataset({
        features: ['messages', 'task', 'category', 'dataset', 'id'],
        num_rows: 10178
    })
    dev: Dataset({
        features: ['messages', 'task', 'category', 'dataset', 'id'],
        num_rows: 1272
    })
    test: Dataset({
        features: ['messages', 'task', 'category', 'dataset', 'id'],
        num_rows: 1273
    })
})

In [35]:
ds10["train"][0]

{'messages': [{'content': '<<Question_Answering: MCQA>>\nYou are performing question answering: read the question and any supporting context or options, and answer it as described in the task-specific instructions that follow, returning the JSON object those instructions specify.',
   'role': 'system'},
  {'content': 'Answer the multiple-choice question by selecting the correct option letter. Return JSON {"answer": <letter>}.\nINPUT: Question: A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. Which of the following is the b

## other

In [36]:
ds11 = datasets.load_from_disk((root / "medmcqa/").as_posix())
ds11["train"][0]

{'messages': [{'content': '<<Question_Answering: MCQA_Explain>>\nYou are performing question answering: read the question and any supporting context or options, and answer it as described in the task-specific instructions that follow, returning the JSON object those instructions specify.',
   'role': 'system'},
  {'content': 'Answer the multiple-choice question and explain your reasoning. Return JSON {"answer": <letter>, "explanation": ...}.\nINPUT: Question: Chronic urethral obstruction due to benign prismatic hyperplasia can lead to the following change in kidney parenchyma\nOptions:\nA. Hyperplasia\nB. Hyperophy\nC. Atrophy\nD. Dyplasia',
   'role': 'user'},
  {'content': '{"answer": "C", "explanation": "Chronic urethral obstruction because of urinary calculi, prostatic hyperophy, tumors, normal pregnancy, tumors, uterine prolapse or functional disorders cause hydronephrosis which by definition is used to describe dilatation of renal pelvis and calculus associated with progressive a

In [38]:
ds12 = datasets.load_from_disk((root / "pubmedqa/").as_posix())
ds12["train"][0]

{'messages': [{'content': '<<Question_Answering: QA_YesNoMaybe>>\nYou are performing question answering: read the question and any supporting context or options, and answer it as described in the task-specific instructions that follow, returning the JSON object those instructions specify.',
   'role': 'system'},
  {'content': 'Answer the research question yes, no, or maybe given the context, and give a long-form answer. Return JSON {"decision": ..., "long_answer": ...}.\nINPUT: Question: Are group 2 innate lymphoid cells ( ILC2s ) increased in chronic rhinosinusitis with nasal polyps or eosinophilia?\nContext:\nChronic rhinosinusitis (CRS) is a heterogeneous disease with an uncertain pathogenesis. Group 2 innate lymphoid cells (ILC2s) represent a recently discovered cell population which has been implicated in driving Th2 inflammation in CRS; however, their relationship with clinical disease characteristics has yet to be investigated.\nThe aim of this study was to identify ILC2s in sin

In [40]:
ds13 = datasets.load_from_disk((root / "emrqa-msquad/").as_posix())
ds13["train"][0]

{'messages': [{'content': '<<Question_Answering: QA_Span_Extraction>>\nYou are performing question answering: read the question and any supporting context or options, and answer it as described in the task-specific instructions that follow, returning the JSON object those instructions specify.',
   'role': 'system'},
  {'content': 'Given the clinical note context and a question, extract the exact answer as a verbatim substring of the context. Return JSON {"answer": <exact substring of the context>}.\nINPUT: Context:\nThe patient was admitted on 5/5/2006 with a history of mechanical fall, with the attending physician being Dr. Clemente Armand Bolstad, with a full code status and disposition of Rehabilitation. Medications on Admission included Amiodarone 100 QD, Colace 100 bid, lasix 40mg QD, Glyburide 5mg bid, Plaquenil 200mg bid, Isordil 20mg tid, Lisinopril 20mg QD, Coumadin 5mg 3dys/week, 2.5mg 4dys/week, Norvasc 10mg QD, Neurontin 300mg TID, with APAP prn. An override was added on 1

In [42]:
ds14 = datasets.load_from_disk((root / "covid_LitCovid").as_posix())
ds14["train"][0]

{'messages': [{'content': '<<Classification: Multilabel_Topic>>\nYou are performing text classification: read the input and assign it the label(s) described in the task-specific instructions that follow, and return them as the JSON object those instructions specify.',
   'role': 'system'},
  {'content': 'Assign all applicable topic labels to this article. Return JSON {"labels": [...]}.\nINPUT: Title: Potential role for tissue factor in the pathogenesis of hypercoagulability associated with in COVID-19.\nAbstract: In December 2019, a new and highly contagious infectious disease emerged in Wuhan, China. The etiologic agent was identified as a novel coronavirus, now known as Severe Acute Syndrome Coronavirus-2 (SARS-CoV-2). Recent research has revealed that virus entry takes place upon the union of the virus S surface protein with the type I transmembrane metallo-carboxypeptidase, angiotensin converting enzyme 2 (ACE-2) identified on epithelial cells of the host respiratory tract. Virus t